# Task 1 — SHAP Feature Importance Stability Across 5 A2C_mod Seeds

**Mục tiêu:** kiểm chứng feature importance của SHAP có ổn định qua 5 lần huấn luyện độc lập (seed 42, 123, 256, 512, 1024) hay không.

**Cấu hình chốt:**
- 5 checkpoint A2C_mod: seed 42 `ckpt-53` (530 ep), các seed còn lại `ckpt-60` (600 ep).
- 3 scenario: EASY (0.5 / 0.010), MEDIUM (1.0 / 0.025), HARD (1.5 / 0.050).
- 2 target: `π` (policy probabilities) và `logits` (pre-softmax).
- 50 test state; micro top-k `k = 10/20/50`; macro top-k `k = 1/2`.
- 1 background cố định: macro 100×3, micro 100×660, seed 42.

**Vì sao dùng cả `π` và `logits`:**
- `π` là quyết định thực tế, so sánh trực tiếp với paper.
- `logits` giữ cấu trúc ưu tiên khi policy bão hòa (seed 256: 88.7% action 13; seed 123: 82.9%; seed 1024: 66.7%), tránh SHAP trên `π` bị gần 0/nhiễu.
- Notebook tính thêm Jaccard/Spearman giữa `π` và `logits` trên cùng seed.

**Output chính:** `output/Task1_Report.md`, bảng CSV, raw NPZ (resume được), và hình trong `output/figs/`.


In [1]:
import os, sys, json, time, random, warnings
from pathlib import Path

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["PYTHONHASHSEED"] = "42"
warnings.filterwarnings("ignore")

BASE = Path(r"C:\GitHub\Q-learning-for-Inventory-Management")
TASK_DIR = BASE / "Feedback 23-11" / "task1"
OUT_DIR = TASK_DIR / "output"
RAW_DIR = OUT_DIR / "raw"
FIG_DIR = OUT_DIR / "figs"
for d in [OUT_DIR, RAW_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DATA_DIR = BASE / "data"
NUM_PRODUCTS = 220
NUM_ACTIONS = 14
NUM_FEATURES_PP = 3
NUM_FEATURES = NUM_PRODUCTS * NUM_FEATURES_PP
ZERO_INVENTORY = 1e-5
A2C_HIDDEN = 32

SEEDS = [42, 123, 256, 512, 1024]
CHECKPOINTS = {
    42:   (BASE / "Feedback 23-11" / "checkpoints_a2c_42", "ckpt-53"),
    123:  (BASE / "Feedback 23-11" / "A2C_mod_123" / "checkpoints", "ckpt-60"),
    256:  (BASE / "Feedback 23-11" / "A2C_mod_256" / "checkpoints", "ckpt-60"),
    512:  (BASE / "Feedback 23-11" / "checkpoints_a2c_512", "ckpt-60"),
    1024: (BASE / "Feedback 23-11" / "checkpoints_a2c_1024", "ckpt-60"),
}
EXTRA_CHECKPOINTS = {
    (256, 76): (BASE / "Feedback 23-11" / "A2C_mod_256" / "checkpoints", "ckpt-76"),
    (1024, 64): (BASE / "Feedback 23-11" / "checkpoints_a2c_1024", "ckpt-64"),
    (42, 50): (BASE / "Feedback 23-11" / "checkpoints_a2c_42", "ckpt-50"),
}

SCENARIOS = {
    "EASY":   {"sales_scale": 0.5, "waste_rate": 0.010},
    "MEDIUM": {"sales_scale": 1.0, "waste_rate": 0.025},
    "HARD":   {"sales_scale": 1.5, "waste_rate": 0.050},
}
TARGETS = ["pi", "logits"]
N_STATES = 50
TOP_K = [10, 20, 50]
MACRO_TOP_K = [1, 2]
BG_SEED = 42
BG_SIZE = 100
ACTION_SPACE = [0.0, 0.005, 0.01, 0.0125, 0.015, 0.0175, 0.02, 0.03, 0.04, 0.08, 0.12, 0.2, 0.5, 1.0]
ACTION_NAMES = [str(a) for a in ACTION_SPACE]

RUN_MACRO = True
RUN_MICRO = True
FORCE_RECOMPUTE = False
RUN_MATCHED_500 = False
RUN_NOISE_FLOOR = False

print("Task dir:", TASK_DIR)
print("Output dir:", OUT_DIR)
print("Seeds:", SEEDS)
print("Scenarios:", list(SCENARIOS))
print("Targets:", TARGETS)


Task dir: C:\GitHub\Q-learning-for-Inventory-Management\Feedback 23-11\task1
Output dir: C:\GitHub\Q-learning-for-Inventory-Management\Feedback 23-11\task1\output
Seeds: [42, 123, 256, 512, 1024]
Scenarios: ['EASY', 'MEDIUM', 'HARD']
Targets: ['pi', 'logits']


In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
import shap
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from IPython.display import display

tf.get_logger().setLevel("ERROR")

def set_global_seed(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_global_seed(42)
print("TF", tf.__version__)
print("SHAP", shap.__version__)
print("NumPy", np.__version__)
print("Pandas", pd.__version__)


TF 2.20.0
SHAP 0.52.0
NumPy 2.2.4
Pandas 2.3.3


In [3]:
class Dense(tf.Module):
    def __init__(self, input_dim, output_size, name=None):
        super().__init__(name=name)
        self.w = tf.Variable(tf.zeros([input_dim, output_size]), name="w")
        self.b = tf.Variable(tf.zeros([output_size]), name="b")

    def __call__(self, x):
        return tf.matmul(x, self.w) + self.b


class Actor(tf.Module):
    def __init__(self, num_features=3, num_actions=14, hidden_size=32, name=None):
        super().__init__(name=name)
        self.layer1 = Dense(num_features, hidden_size)
        self.layer2 = Dense(hidden_size, hidden_size)
        self.layer3 = Dense(hidden_size, hidden_size)
        self.layer4 = Dense(hidden_size, num_actions)

    def logits(self, state):
        x = tf.nn.relu(self.layer1(state))
        x = tf.nn.relu(self.layer2(x))
        x = tf.nn.relu(self.layer3(x))
        return self.layer4(x)

    def __call__(self, state):
        return tf.nn.softmax(self.logits(state))


_actor_global = None
print("Actor 3 -> 32 -> 32 -> 32 -> 14, dropout=0.0")


Actor 3 -> 32 -> 32 -> 32 -> 14, dropout=0.0


In [4]:
def _parse_single(serialized, key, n):
    features = tf.io.parse_single_example(
        serialized, {key: tf.io.FixedLenFeature([n], tf.float32)}
    )
    return features[key]


def _load_vector(path, key, n):
    ds = tf.data.TFRecordDataset(str(path)).map(lambda s: _parse_single(s, key, n))
    return next(iter(ds)).numpy().astype(np.float32)


def _load_matrix(path, key, n):
    ds = tf.data.TFRecordDataset(str(path)).map(lambda s: _parse_single(s, key, n))
    return np.array([x.numpy() for x in ds], dtype=np.float32)


capacity = _load_vector(DATA_DIR / "capacity.tfrecords", "capacity", NUM_PRODUCTS)
x_init = _load_vector(DATA_DIR / "stock.tfrecords", "stock", NUM_PRODUCTS)
all_sales_raw = _load_matrix(DATA_DIR / "test.tfrecords", "sales", NUM_PRODUCTS)
all_sales = all_sales_raw / capacity[None, :]
assert all_sales.shape[0] >= N_STATES


def build_states_660(scenario, n=N_STATES):
    cfg = SCENARIOS[scenario]
    sales = all_sales[:n] * cfg["sales_scale"]
    states = np.zeros((n, NUM_FEATURES), dtype=np.float32)
    states[:, :NUM_PRODUCTS] = x_init[None, :]
    states[:, NUM_PRODUCTS:2 * NUM_PRODUCTS] = sales
    states[:, 2 * NUM_PRODUCTS:] = x_init[None, :] * cfg["waste_rate"]
    return states


def to_macro_3(states_660):
    inv = states_660[:, :NUM_PRODUCTS].mean(axis=1)
    sales = states_660[:, NUM_PRODUCTS:2 * NUM_PRODUCTS].mean(axis=1)
    waste = states_660[:, 2 * NUM_PRODUCTS:].mean(axis=1)
    return np.column_stack([inv, sales, waste]).astype(np.float32)


def generate_background_3(n=BG_SIZE, seed=BG_SEED):
    rng = np.random.default_rng(seed)
    inv = rng.uniform(0, 1, n)
    sales = rng.uniform(0, 1, n)
    waste = np.clip(0.025 * inv + rng.normal(0, 0.005, n), 0, 0.1)
    return np.column_stack([inv, sales, waste]).astype(np.float32)


def generate_background_660(n=BG_SIZE, seed=BG_SEED):
    rng = np.random.default_rng(seed)
    bg = np.zeros((n, NUM_FEATURES), dtype=np.float32)
    bg[:, :NUM_PRODUCTS] = rng.uniform(0, 1, (n, NUM_PRODUCTS))
    bg[:, NUM_PRODUCTS:2 * NUM_PRODUCTS] = rng.uniform(0, 1, (n, NUM_PRODUCTS))
    bg[:, 2 * NUM_PRODUCTS:] = np.clip(
        0.025 * bg[:, :NUM_PRODUCTS] + rng.normal(0, 0.005, (n, NUM_PRODUCTS)), 0, 0.1
    )
    return bg.astype(np.float32)


STATES = {sc: build_states_660(sc) for sc in SCENARIOS}
STATES_MACRO = {sc: to_macro_3(STATES[sc]) for sc in SCENARIOS}
BG_MACRO = generate_background_3()
BG_MICRO = generate_background_660()

print("capacity", capacity.shape, "x_init", x_init.shape)
print("all_sales", all_sales.shape)
for sc in SCENARIOS:
    print(sc, STATES[sc].shape, STATES_MACRO[sc].shape)
print("BG_MACRO", BG_MACRO.shape, "BG_MICRO", BG_MICRO.shape)


capacity (220,) x_init (220,)
all_sales (504, 220)
EASY (50, 660) (50, 3)
MEDIUM (50, 660) (50, 3)
HARD (50, 660) (50, 3)
BG_MACRO (100, 3) BG_MICRO (100, 660)


In [5]:
def load_actor(seed, checkpoint=None):
    global _actor_global
    ckpt_dir, default_ckpt = CHECKPOINTS[seed]
    ckpt_name = checkpoint or default_ckpt
    prefix = str(Path(ckpt_dir) / ckpt_name)
    actor = Actor(NUM_FEATURES_PP, NUM_ACTIONS, A2C_HIDDEN)
    _ = actor(tf.zeros([1, NUM_FEATURES_PP], dtype=tf.float32))
    status = tf.train.Checkpoint(actor=actor).restore(prefix)
    status.expect_partial()
    _actor_global = actor
    return actor, prefix


def _states_660_to_pp(states_660):
    n = states_660.shape[0]
    return states_660.reshape(n, 3, NUM_PRODUCTS).transpose(0, 2, 1).reshape(n * NUM_PRODUCTS, 3)


def action_stats(actor, states_660):
    s3 = _states_660_to_pp(states_660)
    probs = actor(tf.constant(s3, dtype=tf.float32)).numpy()
    acts = probs.argmax(axis=1)
    ent = -(probs * np.log(np.maximum(probs, 1e-12))).sum(axis=1)
    hist = np.bincount(acts, minlength=NUM_ACTIONS) / len(acts)
    return {
        "n_sku_states": int(len(acts)),
        "mean_action": float(acts.mean()),
        "std_action": float(acts.std()),
        "frac_action_13": float(hist[13]),
        "mean_entropy": float(ent.mean()),
        "mean_max_prob": float(probs.max(axis=1).mean()),
        **{f"action_{i}": float(hist[i]) for i in range(NUM_ACTIONS)},
    }


def rollout_metrics(actor, scenario, steps=100):
    cfg = SCENARIOS[scenario]
    x = x_init.copy()
    rewards, stockouts, wastes, overstocks, quans, actions = [], [], [], [], [], []
    for t in range(min(steps, all_sales.shape[0])):
        sales = all_sales[t] * cfg["sales_scale"]
        waste_val = cfg["waste_rate"] * x
        state = np.column_stack([x, sales, waste_val]).astype(np.float32)
        probs = actor(tf.constant(state, dtype=tf.float32)).numpy()
        a = probs.argmax(axis=1)
        u = np.array(ACTION_SPACE, dtype=np.float32)[a]
        overstock = np.maximum(0.0, x + u - 1.0)
        x_u = np.minimum(1.0, x + u)
        x_prime = np.maximum(0.0, x_u - sales)
        z = (x < ZERO_INVENTORY).astype(np.float32)
        quan = np.full(NUM_PRODUCTS, np.quantile(x, 0.95) - np.quantile(x, 0.05), dtype=np.float32)
        r = 1.0 - z - overstock - cfg["waste_rate"] * x - quan
        rewards.append(r.mean())
        stockouts.append(z.mean())
        wastes.append((cfg["waste_rate"] * x).mean())
        overstocks.append(overstock.mean())
        quans.append(quan.mean())
        actions.append(a.mean())
        x = x_prime
    return {
        "reward": float(np.mean(rewards)),
        "stockout": float(np.mean(stockouts)),
        "waste": float(np.mean(wastes)),
        "overstock": float(np.mean(overstocks)),
        "quan": float(np.mean(quans)),
        "mean_action": float(np.mean(actions)),
        "steps": int(min(steps, all_sales.shape[0])),
    }


ckpt_rows = []
diag_rows = []
rollout_rows = []
for seed in SEEDS:
    ckpt_dir, ckpt_name = CHECKPOINTS[seed]
    ckpt_id = int(ckpt_name.split("-")[1])
    actor, prefix = load_actor(seed)
    ckpt_rows.append({
        "seed": seed,
        "checkpoint_dir": str(ckpt_dir),
        "checkpoint": ckpt_name,
        "step": ckpt_id * 330,
        "episode": ckpt_id * 10,
        "prefix": prefix,
    })
    for sc in SCENARIOS:
        st = action_stats(actor, STATES[sc])
        diag_rows.append({"seed": seed, "scenario": sc, **st})
        rm = rollout_metrics(actor, sc)
        rollout_rows.append({"seed": seed, "scenario": sc, **rm})
    print(f"seed {seed}: loaded {prefix}")

df_ckpt = pd.DataFrame(ckpt_rows)
df_diag = pd.DataFrame(diag_rows)
df_rollout = pd.DataFrame(rollout_rows)
df_ckpt.to_csv(OUT_DIR / "checkpoints_used.csv", index=False)
df_diag.to_csv(OUT_DIR / "collapse_diagnostics.csv", index=False)
df_rollout.to_csv(OUT_DIR / "rollout_metrics.csv", index=False)

display(df_ckpt)
display(df_diag[["seed", "scenario", "mean_action", "frac_action_13", "mean_entropy", "mean_max_prob"]])
display(df_rollout)


seed 42: loaded C:\GitHub\Q-learning-for-Inventory-Management\Feedback 23-11\checkpoints_a2c_42\ckpt-53


seed 123: loaded C:\GitHub\Q-learning-for-Inventory-Management\Feedback 23-11\A2C_mod_123\checkpoints\ckpt-60


seed 256: loaded C:\GitHub\Q-learning-for-Inventory-Management\Feedback 23-11\A2C_mod_256\checkpoints\ckpt-60


seed 512: loaded C:\GitHub\Q-learning-for-Inventory-Management\Feedback 23-11\checkpoints_a2c_512\ckpt-60


seed 1024: loaded C:\GitHub\Q-learning-for-Inventory-Management\Feedback 23-11\checkpoints_a2c_1024\ckpt-60


,seed,checkpoint_dir,checkpoint,step,episode,prefix
0,42,C:\GitHub\Q-learning-for-Inventory-Management\...,ckpt-53,17490,530,C:\GitHub\Q-learning-for-Inventory-Management\...
1,123,C:\GitHub\Q-learning-for-Inventory-Management\...,ckpt-60,19800,600,C:\GitHub\Q-learning-for-Inventory-Management\...
2,256,C:\GitHub\Q-learning-for-Inventory-Management\...,ckpt-60,19800,600,C:\GitHub\Q-learning-for-Inventory-Management\...
3,512,C:\GitHub\Q-learning-for-Inventory-Management\...,ckpt-60,19800,600,C:\GitHub\Q-learning-for-Inventory-Management\...
4,1024,C:\GitHub\Q-learning-for-Inventory-Management\...,ckpt-60,19800,600,C:\GitHub\Q-learning-for-Inventory-Management\...


,seed,scenario,mean_action,frac_action_13,mean_entropy,mean_max_prob
0,42,EASY,8.794000,0.000000,0.000686,0.999753
1,42,MEDIUM,8.978000,0.000000,0.000786,0.999644
2,42,HARD,9.106000,0.000000,0.000357,0.999792
3,123,EASY,11.158000,0.832545,0.013230,0.994940
4,123,MEDIUM,11.114000,0.828545,0.010734,0.996038
5,123,HARD,11.140727,0.830727,0.011008,0.995974
6,256,EASY,12.424091,0.884818,0.009605,0.994962
7,256,MEDIUM,12.435000,0.887000,0.009456,0.994407
8,256,HARD,12.385000,0.877000,0.009267,0.994491
9,512,EASY,9.371727,0.000000,0.006868,0.996639


,seed,scenario,reward,stockout,waste,overstock,quan,mean_action,steps
0,42,EASY,0.794103,0.000000,0.008322,0.000873,0.196703,1.596000,100
1,42,MEDIUM,0.765039,0.000091,0.020363,0.004212,0.210295,3.131500,100
2,42,HARD,0.633208,0.001727,0.039432,0.006613,0.319018,4.591000,100
3,123,EASY,-0.879068,0.054636,0.008260,0.822394,0.993778,11.315000,100
4,123,MEDIUM,-0.876091,0.067818,0.019901,0.792871,0.995500,11.214318,100
5,123,HARD,-0.859269,0.076500,0.037653,0.749204,0.995912,10.940545,100
6,256,EASY,-0.114718,0.000045,0.009645,0.963933,0.141095,12.966136,100
7,256,MEDIUM,-0.306697,0.013273,0.022796,0.910559,0.360070,12.811591,100
8,256,HARD,-0.608984,0.024273,0.043740,0.872990,0.667981,12.731818,100
9,512,EASY,0.778143,0.000000,0.009454,0.051095,0.161308,9.003409,100


In [6]:
def _macro_pi(x):
    x = np.asarray(x, dtype=np.float32)
    return _actor_global(tf.constant(x, dtype=tf.float32)).numpy()


def _macro_logits(x):
    x = np.asarray(x, dtype=np.float32)
    return _actor_global.logits(tf.constant(x, dtype=tf.float32)).numpy()


def _micro_probs(X):
    X = np.asarray(X, dtype=np.float32)
    B = X.shape[0]
    s3 = X.reshape(B, 3, NUM_PRODUCTS).transpose(0, 2, 1).reshape(B * NUM_PRODUCTS, 3)
    probs = _actor_global(tf.constant(s3, dtype=tf.float32))
    probs = tf.reshape(probs, [B, NUM_PRODUCTS, NUM_ACTIONS])
    return tf.reduce_mean(probs, axis=1).numpy()


def _micro_logits(X):
    X = np.asarray(X, dtype=np.float32)
    B = X.shape[0]
    s3 = X.reshape(B, 3, NUM_PRODUCTS).transpose(0, 2, 1).reshape(B * NUM_PRODUCTS, 3)
    logits = _actor_global.logits(tf.constant(s3, dtype=tf.float32))
    logits = tf.reshape(logits, [B, NUM_PRODUCTS, NUM_ACTIONS])
    return tf.reduce_mean(logits, axis=1).numpy()


print("SHAP wrappers ready: macro pi/logits, micro pi/logits")


SHAP wrappers ready: macro pi/logits, micro pi/logits


In [7]:
def _normalize_macro(sv, n):
    if isinstance(sv, list):
        arr = np.stack([np.asarray(a, dtype=np.float32) for a in sv], axis=-1)
    else:
        arr = np.asarray(sv, dtype=np.float32)
        if arr.ndim == 2:
            arr = arr[..., None]
        elif arr.ndim == 3 and arr.shape[0] == NUM_ACTIONS and arr.shape[1] == n:
            arr = arr.transpose(1, 2, 0)
    return arr


def _normalize_micro(sv):
    if isinstance(sv, list):
        arr = np.stack([np.asarray(a, dtype=np.float32) for a in sv], axis=-1)
    else:
        arr = np.asarray(sv, dtype=np.float32)
        if arr.ndim == 2:
            arr = arr[..., None]
    return arr


def run_macro(seed, scenario, target, force=False):
    out = RAW_DIR / f"macro_{target}_{scenario}_seed{seed}.npz"
    if out.exists() and not force:
        return out
    load_actor(seed)
    predict = _macro_pi if target == "pi" else _macro_logits
    X = STATES_MACRO[scenario]
    explainer = shap.KernelExplainer(predict, BG_MACRO)
    np.random.seed(BG_SEED)
    sv = explainer.shap_values(X)
    arr = _normalize_macro(sv, len(X))
    np.savez_compressed(out, values=arr, X=X, background=BG_MACRO)
    return out


def run_micro(seed, scenario, target, force=False):
    out = RAW_DIR / f"micro_{target}_{scenario}_seed{seed}.npz"
    if out.exists() and not force:
        return out
    load_actor(seed)
    predict = _micro_probs if target == "pi" else _micro_logits
    X = STATES[scenario]
    masker = shap.maskers.Partition(BG_MICRO, max_samples=BG_SIZE)
    explainer = shap.PartitionExplainer(predict, masker)
    np.random.seed(BG_SEED)
    sv = explainer(X)
    arr = _normalize_micro(sv.values if hasattr(sv, "values") else sv)
    np.savez_compressed(out, values=arr, X=X, background=BG_MICRO)
    return out


print("SHAP runners ready (resume enabled)")


SHAP runners ready (resume enabled)


In [8]:
if RUN_MACRO:
    for target in TARGETS:
        for scenario in SCENARIOS:
            for seed in SEEDS:
                t0 = time.time()
                p = run_macro(seed, scenario, target, force=FORCE_RECOMPUTE)
                print(f"macro {target:6s} {scenario:6s} seed {seed:4d} -> {p.name} ({time.time()-t0:.1f}s)")

if RUN_MICRO:
    for target in TARGETS:
        for scenario in SCENARIOS:
            for seed in SEEDS:
                t0 = time.time()
                p = run_micro(seed, scenario, target, force=FORCE_RECOMPUTE)
                print(f"micro {target:6s} {scenario:6s} seed {seed:4d} -> {p.name} ({time.time()-t0:.1f}s)")

print("SHAP run complete")


macro pi     EASY   seed   42 -> macro_pi_EASY_seed42.npz (0.0s)
macro pi     EASY   seed  123 -> macro_pi_EASY_seed123.npz (0.0s)
macro pi     EASY   seed  256 -> macro_pi_EASY_seed256.npz (0.0s)
macro pi     EASY   seed  512 -> macro_pi_EASY_seed512.npz (0.0s)
macro pi     EASY   seed 1024 -> macro_pi_EASY_seed1024.npz (0.0s)
macro pi     MEDIUM seed   42 -> macro_pi_MEDIUM_seed42.npz (0.0s)
macro pi     MEDIUM seed  123 -> macro_pi_MEDIUM_seed123.npz (0.0s)
macro pi     MEDIUM seed  256 -> macro_pi_MEDIUM_seed256.npz (0.0s)
macro pi     MEDIUM seed  512 -> macro_pi_MEDIUM_seed512.npz (0.0s)
macro pi     MEDIUM seed 1024 -> macro_pi_MEDIUM_seed1024.npz (0.0s)
macro pi     HARD   seed   42 -> macro_pi_HARD_seed42.npz (0.0s)
macro pi     HARD   seed  123 -> macro_pi_HARD_seed123.npz (0.0s)
macro pi     HARD   seed  256 -> macro_pi_HARD_seed256.npz (0.0s)
macro pi     HARD   seed  512 -> macro_pi_HARD_seed512.npz (0.0s)
macro pi     HARD   seed 1024 -> macro_pi_HARD_seed1024.npz (0.0s)


In [9]:
def mean_abs_macro(arr):
    return np.mean(np.abs(arr), axis=(0, 2))


def mean_abs_micro(arr):
    return np.mean(np.abs(arr), axis=(0, 2))


macro_rows = []
micro_rows = []
for target in TARGETS:
    for scenario in SCENARIOS:
        for seed in SEEDS:
            p = RAW_DIR / f"macro_{target}_{scenario}_seed{seed}.npz"
            if p.exists():
                arr = np.load(p)["values"]
                imp = mean_abs_macro(arr)
                for j, name in enumerate(["Inventory", "Demand", "Waste"]):
                    macro_rows.append({
                        "target": target, "scenario": scenario, "seed": seed,
                        "feature_idx": j, "feature": name,
                        "mean_abs_shap": float(imp[j]),
                    })
            p = RAW_DIR / f"micro_{target}_{scenario}_seed{seed}.npz"
            if p.exists():
                arr = np.load(p)["values"]
                imp = mean_abs_micro(arr)
                for j in range(NUM_FEATURES):
                    group = "inventory" if j < NUM_PRODUCTS else ("sales" if j < 2 * NUM_PRODUCTS else "waste")
                    sku = j % NUM_PRODUCTS
                    micro_rows.append({
                        "target": target, "scenario": scenario, "seed": seed,
                        "feature_idx": j, "feature": f"{group}_sku{sku}",
                        "group": group, "sku": sku,
                        "mean_abs_shap": float(imp[j]),
                    })

df_macro_imp = pd.DataFrame(macro_rows)
df_micro_imp = pd.DataFrame(micro_rows)
df_macro_imp.to_csv(OUT_DIR / "shap_macro_importance.csv", index=False)
df_micro_imp.to_csv(OUT_DIR / "shap_micro_importance.csv", index=False)
print("macro rows", len(df_macro_imp), "micro rows", len(df_micro_imp))
display(df_macro_imp.head())
display(df_micro_imp.head())


macro rows 90 micro rows 19800


,target,scenario,seed,feature_idx,feature,mean_abs_shap
0,pi,EASY,42,0,Inventory,0.015290
1,pi,EASY,42,1,Demand,0.012574
2,pi,EASY,42,2,Waste,0.000141
3,pi,EASY,123,0,Inventory,0.027264
4,pi,EASY,123,1,Demand,0.008062


,target,scenario,seed,feature_idx,feature,group,sku,mean_abs_shap
0,pi,EASY,42,0,inventory_sku0,inventory,0,0.000034
1,pi,EASY,42,1,inventory_sku1,inventory,1,0.000034
2,pi,EASY,42,2,inventory_sku2,inventory,2,0.000034
3,pi,EASY,42,3,inventory_sku3,inventory,3,0.000034
4,pi,EASY,42,4,inventory_sku4,inventory,4,0.000034


In [10]:
def jaccard(a, b):
    sa, sb = set(int(x) for x in a), set(int(x) for x in b)
    if not (sa | sb):
        return 0.0
    return len(sa & sb) / len(sa | sb)


def topk_idx(values, k):
    values = np.asarray(values, dtype=np.float64)
    k = min(k, len(values))
    return list(np.argsort(values)[::-1][:k])


def _safe_spearman(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    if len(a) < 2:
        return 0.0
    r = spearmanr(a, b).correlation
    return 0.0 if np.isnan(r) else float(r)


def pairwise_stability(imp_by_seed, k_list):
    seeds = [s for s in SEEDS if s in imp_by_seed]
    rows = []
    for i, sa in enumerate(seeds):
        for sb in seeds[i + 1:]:
            a = np.asarray(imp_by_seed[sa], dtype=np.float64)
            b = np.asarray(imp_by_seed[sb], dtype=np.float64)
            rho_all = _safe_spearman(a, b)
            for k in k_list:
                ia, ib = topk_idx(a, k), topk_idx(b, k)
                jc = jaccard(ia, ib)
                union = sorted(set(ia) | set(ib))
                rho_top = _safe_spearman([a[j] for j in union], [b[j] for j in union])
                rows.append({
                    "seed_a": sa, "seed_b": sb, "k": k,
                    "jaccard": float(jc),
                    "spearman_all": float(rho_all),
                    "spearman_topk": float(rho_top),
                })
    return rows


def get_imp(df, target, scenario):
    sub = df[(df["target"] == target) & (df["scenario"] == scenario)]
    out = {}
    for seed in SEEDS:
        s = sub[sub["seed"] == seed].sort_values("feature_idx")
        if len(s):
            out[seed] = s["mean_abs_shap"].values
    return out


pair_rows = []
for level, df in [("macro", df_macro_imp), ("micro", df_micro_imp)]:
    ks = MACRO_TOP_K if level == "macro" else TOP_K
    for target in TARGETS:
        for scenario in SCENARIOS:
            imp = get_imp(df, target, scenario)
            if len(imp) < 2:
                continue
            for subset_name, subset in [
                ("all", SEEDS),
                ("excl_256", [s for s in SEEDS if s != 256]),
                ("stable_pair", [42, 512]),
            ]:
                sub_imp = {s: imp[s] for s in subset if s in imp}
                if len(sub_imp) < 2:
                    continue
                for r in pairwise_stability(sub_imp, ks):
                    pair_rows.append({
                        "level": level, "target": target, "scenario": scenario,
                        "subset": subset_name, **r,
                    })

df_pairs = pd.DataFrame(pair_rows)
df_pairs.to_csv(OUT_DIR / "stability_pairwise.csv", index=False)

if len(df_pairs):
    df_summary = (
        df_pairs.groupby(["level", "target", "scenario", "subset", "k"])
        .agg(
            jaccard_mean=("jaccard", "mean"), jaccard_sd=("jaccard", "std"),
            spearman_all_mean=("spearman_all", "mean"), spearman_all_sd=("spearman_all", "std"),
            spearman_topk_mean=("spearman_topk", "mean"), spearman_topk_sd=("spearman_topk", "std"),
            n_pairs=("jaccard", "size"),
        )
        .reset_index()
    )
else:
    df_summary = pd.DataFrame()
df_summary.to_csv(OUT_DIR / "stability_summary.csv", index=False)
display(df_summary)


,level,target,scenario,subset,k,jaccard_mean,jaccard_sd,spearman_all_mean,spearman_all_sd,spearman_topk_mean,spearman_topk_sd,n_pairs
0,macro,logits,EASY,all,1,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,10
1,macro,logits,EASY,all,2,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,10
2,macro,logits,EASY,excl_256,1,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,6
3,macro,logits,EASY,excl_256,2,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,6
4,macro,logits,EASY,stable_pair,1,1.000000,NaN,1.000000,NaN,0.000000,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...
85,micro,pi,MEDIUM,excl_256,20,0.869565,0.142884,0.933940,0.072278,0.874888,0.031468,6
86,micro,pi,MEDIUM,excl_256,50,0.862069,0.151096,0.933940,0.072278,0.948722,0.038501,6
87,micro,pi,MEDIUM,stable_pair,10,1.000000,NaN,0.999909,NaN,-0.073171,NaN,1
88,micro,pi,MEDIUM,stable_pair,20,1.000000,NaN,0.999909,NaN,0.841298,NaN,1


In [11]:
def mean_sd_table(df, group_cols):
    if len(df) == 0:
        return pd.DataFrame()
    g = df.groupby(group_cols)["mean_abs_shap"]
    out = g.agg(mean="mean", sd="std", n="count").reset_index()
    out["cv"] = out["sd"] / out["mean"].replace(0, np.nan)
    return out


def rank_sd_table(df, id_cols):
    if len(df) == 0:
        return pd.DataFrame()
    rows = []
    for keys, g in df.groupby(id_cols):
        piv = g.pivot_table(index="seed", values="mean_abs_shap", aggfunc="first")
        ranks = piv.rank(ascending=False, method="average")
        rows.append({**dict(zip(id_cols, keys)), "rank_mean": float(ranks.mean().iloc[0]), "rank_sd": float(ranks.std().iloc[0])})
    return pd.DataFrame(rows)


df_macro_stats = mean_sd_table(df_macro_imp, ["target", "scenario", "feature_idx", "feature"])
df_micro_stats = mean_sd_table(df_micro_imp, ["target", "scenario", "feature_idx", "feature", "group", "sku"])
df_macro_stats.to_csv(OUT_DIR / "mean_sd_macro.csv", index=False)
df_micro_stats.to_csv(OUT_DIR / "mean_sd_micro.csv", index=False)

if len(df_macro_stats):
    df_macro_rank = rank_sd_table(df_macro_imp, ["target", "scenario", "feature_idx", "feature"])
    df_macro_rank.to_csv(OUT_DIR / "rank_sd_macro.csv", index=False)
if len(df_micro_stats):
    df_micro_rank = rank_sd_table(df_micro_imp, ["target", "scenario", "feature_idx", "feature", "group", "sku"])
    df_micro_rank.to_csv(OUT_DIR / "rank_sd_micro.csv", index=False)

display(df_macro_stats)
if len(df_micro_stats):
    display(df_micro_stats.sort_values(["target", "scenario", "mean"], ascending=[True, True, False]).groupby(["target", "scenario"]).head(10))


,target,scenario,feature_idx,feature,mean,sd,n,cv
0,logits,EASY,0,Inventory,6.880954,4.635867,5,0.673724
1,logits,EASY,1,Demand,26.339214,7.384950,5,0.280379
2,logits,EASY,2,Waste,0.343828,0.141173,5,0.410594
3,logits,HARD,0,Inventory,7.147851,4.260627,5,0.596071
4,logits,HARD,1,Demand,23.994487,7.649176,5,0.318789
5,logits,HARD,2,Waste,0.536820,0.199936,5,0.372445
6,logits,MEDIUM,0,Inventory,7.006348,4.449703,5,0.635096
7,logits,MEDIUM,1,Demand,25.092422,7.573585,5,0.301828
8,logits,MEDIUM,2,Waste,0.053883,0.019045,5,0.353457
9,pi,EASY,0,Inventory,0.023020,0.009526,5,0.413823


,target,scenario,feature_idx,feature,group,sku,mean,sd,n,cv
286,logits,EASY,286,sales_sku66,sales,66,0.155493,0.023583,5,0.151662
234,logits,EASY,234,sales_sku14,sales,14,0.152153,0.040060,5,0.263290
348,logits,EASY,348,sales_sku128,sales,128,0.151665,0.021524,5,0.141918
291,logits,EASY,291,sales_sku71,sales,71,0.148455,0.035950,5,0.242158
382,logits,EASY,382,sales_sku162,sales,162,0.133010,0.043514,5,0.327148
285,logits,EASY,285,sales_sku65,sales,65,0.128277,0.043375,5,0.338135
224,logits,EASY,224,sales_sku4,sales,4,0.120484,0.024518,5,0.203500
279,logits,EASY,279,sales_sku59,sales,59,0.120484,0.024518,5,0.203500
256,logits,EASY,256,sales_sku36,sales,36,0.118488,0.040218,5,0.339428
0,logits,EASY,0,inventory_sku0,inventory,0,0.039119,0.008913,5,0.227849


In [12]:
cross_rows = []
for level, df in [("macro", df_macro_imp), ("micro", df_micro_imp)]:
    ks = MACRO_TOP_K if level == "macro" else TOP_K
    for scenario in SCENARIOS:
        for seed in SEEDS:
            pi = df[(df["target"] == "pi") & (df["scenario"] == scenario) & (df["seed"] == seed)].sort_values("feature_idx")["mean_abs_shap"].values
            lg = df[(df["target"] == "logits") & (df["scenario"] == scenario) & (df["seed"] == seed)].sort_values("feature_idx")["mean_abs_shap"].values
            if len(pi) == 0 or len(lg) == 0:
                continue
            rho = _safe_spearman(pi, lg)
            for k in ks:
                cross_rows.append({
                    "level": level, "scenario": scenario, "seed": seed, "k": k,
                    "jaccard": jaccard(topk_idx(pi, k), topk_idx(lg, k)),
                    "spearman_all": rho,
                })
df_cross = pd.DataFrame(cross_rows)
df_cross.to_csv(OUT_DIR / "cross_target_stability.csv", index=False)
display(df_cross)


,level,scenario,seed,k,jaccard,spearman_all
0,macro,EASY,42,1,0.000000,0.500000
1,macro,EASY,42,2,1.000000,0.500000
2,macro,EASY,123,1,0.000000,0.500000
3,macro,EASY,123,2,1.000000,0.500000
4,macro,EASY,256,1,0.000000,0.500000
...,...,...,...,...,...,...
70,micro,HARD,512,20,1.000000,0.999974
71,micro,HARD,512,50,1.000000,0.999974
72,micro,HARD,1024,10,0.818182,0.867966
73,micro,HARD,1024,20,0.818182,0.867966


In [13]:
def plot_macro_mean_sd():
    if len(df_macro_stats) == 0:
        return []
    paths = []
    fig, axes = plt.subplots(len(TARGETS), len(SCENARIOS), figsize=(12, 6), squeeze=False)
    for i, target in enumerate(TARGETS):
        for j, scenario in enumerate(SCENARIOS):
            ax = axes[i, j]
            sub = df_macro_stats[(df_macro_stats["target"] == target) & (df_macro_stats["scenario"] == scenario)]
            x = np.arange(len(sub))
            ax.bar(x, sub["mean"], yerr=sub["sd"], capsize=3, color=["#4C72B0", "#DD8452", "#55A868"][:len(sub)])
            ax.set_xticks(x)
            ax.set_xticklabels(sub["feature"])
            ax.set_title(f"{target} / {scenario}")
            ax.set_ylabel("mean |SHAP|")
    fig.tight_layout()
    p = FIG_DIR / "macro_mean_sd.png"
    fig.savefig(p, dpi=150)
    plt.close(fig)
    paths.append(p)
    return paths


def plot_micro_top20():
    if len(df_micro_stats) == 0:
        return []
    paths = []
    for target in TARGETS:
        for scenario in SCENARIOS:
            sub = df_micro_stats[(df_micro_stats["target"] == target) & (df_micro_stats["scenario"] == scenario)].sort_values("mean", ascending=False).head(20)
            if len(sub) == 0:
                continue
            fig, ax = plt.subplots(figsize=(10, 6))
            colors = sub["group"].map({"inventory": "#4C72B0", "sales": "#DD8452", "waste": "#55A868"})
            ax.barh(range(len(sub))[::-1], sub["mean"], xerr=sub["sd"], color=colors, capsize=2)
            ax.set_yticks(range(len(sub))[::-1])
            ax.set_yticklabels(sub["feature"], fontsize=7)
            ax.set_xlabel("mean |SHAP|")
            ax.set_title(f"Top-20 micro |SHAP| - {target} / {scenario}")
            fig.tight_layout()
            p = FIG_DIR / f"micro_top20_{target}_{scenario}.png"
            fig.savefig(p, dpi=150)
            plt.close(fig)
            paths.append(p)
    return paths


def plot_jaccard_heatmaps():
    if len(df_pairs) == 0:
        return []
    paths = []
    for level in ["macro", "micro"]:
        ks = MACRO_TOP_K if level == "macro" else TOP_K
        for target in TARGETS:
            for scenario in SCENARIOS:
                for k in ks:
                    sub = df_pairs[(df_pairs["level"] == level) & (df_pairs["target"] == target) & (df_pairs["scenario"] == scenario) & (df_pairs["subset"] == "all") & (df_pairs["k"] == k)]
                    if len(sub) == 0:
                        continue
                    mat = np.full((len(SEEDS), len(SEEDS)), np.nan)
                    for _, r in sub.iterrows():
                        i = SEEDS.index(int(r["seed_a"]))
                        j = SEEDS.index(int(r["seed_b"]))
                        mat[i, j] = r["jaccard"]
                        mat[j, i] = r["jaccard"]
                    np.fill_diagonal(mat, 1.0)
                    fig, ax = plt.subplots(figsize=(5, 4))
                    im = ax.imshow(mat, vmin=0, vmax=1, cmap="viridis")
                    ax.set_xticks(range(len(SEEDS)))
                    ax.set_xticklabels(SEEDS)
                    ax.set_yticks(range(len(SEEDS)))
                    ax.set_yticklabels(SEEDS)
                    for i in range(len(SEEDS)):
                        for j in range(len(SEEDS)):
                            if not np.isnan(mat[i, j]):
                                ax.text(j, i, f"{mat[i, j]:.2f}", ha="center", va="center", color="w" if mat[i, j] < 0.5 else "k", fontsize=8)
                    ax.set_title(f"Jaccard {level} k={k} - {target}/{scenario}")
                    fig.colorbar(im, ax=ax)
                    fig.tight_layout()
                    p = FIG_DIR / f"jaccard_{level}_{target}_{scenario}_k{k}.png"
                    fig.savefig(p, dpi=150)
                    plt.close(fig)
                    paths.append(p)
    return paths


def plot_action_distribution():
    paths = []
    for scenario in SCENARIOS:
        sub = df_diag[df_diag["scenario"] == scenario].set_index("seed")
        mat = sub[[f"action_{i}" for i in range(NUM_ACTIONS)]].values
        fig, ax = plt.subplots(figsize=(10, 3))
        im = ax.imshow(mat, aspect="auto", cmap="magma")
        ax.set_yticks(range(len(sub.index)))
        ax.set_yticklabels(sub.index)
        ax.set_xticks(range(NUM_ACTIONS))
        ax.set_xticklabels(ACTION_NAMES, rotation=45, fontsize=7)
        ax.set_title(f"Action distribution - {scenario}")
        fig.colorbar(im, ax=ax)
        fig.tight_layout()
        p = FIG_DIR / f"action_distribution_{scenario}.png"
        fig.savefig(p, dpi=150)
        plt.close(fig)
        paths.append(p)
    return paths


def plot_jaccard_vs_k():
    if len(df_summary) == 0:
        return []
    paths = []
    for level in ["macro", "micro"]:
        for target in TARGETS:
            fig, ax = plt.subplots(figsize=(7, 4))
            for scenario in SCENARIOS:
                sub = df_summary[(df_summary["level"] == level) & (df_summary["target"] == target) & (df_summary["scenario"] == scenario) & (df_summary["subset"] == "all")]
                if len(sub) == 0:
                    continue
                ax.errorbar(sub["k"], sub["jaccard_mean"], yerr=sub["jaccard_sd"], marker="o", capsize=3, label=scenario)
            ax.set_xlabel("k")
            ax.set_ylabel("Jaccard mean +/- SD")
            ax.set_ylim(0, 1.05)
            ax.set_title(f"{level} / {target}")
            ax.legend()
            ax.grid(alpha=0.3)
            fig.tight_layout()
            p = FIG_DIR / f"jaccard_vs_k_{level}_{target}.png"
            fig.savefig(p, dpi=150)
            plt.close(fig)
            paths.append(p)
    return paths


fig_paths = []
fig_paths += plot_macro_mean_sd()
fig_paths += plot_micro_top20()
fig_paths += plot_jaccard_heatmaps()
fig_paths += plot_action_distribution()
fig_paths += plot_jaccard_vs_k()
print("Saved figures:", len(fig_paths))
for p in fig_paths:
    print(" ", p.name)


Saved figures: 44
  macro_mean_sd.png
  micro_top20_pi_EASY.png
  micro_top20_pi_MEDIUM.png
  micro_top20_pi_HARD.png
  micro_top20_logits_EASY.png
  micro_top20_logits_MEDIUM.png
  micro_top20_logits_HARD.png
  jaccard_macro_pi_EASY_k1.png
  jaccard_macro_pi_EASY_k2.png
  jaccard_macro_pi_MEDIUM_k1.png
  jaccard_macro_pi_MEDIUM_k2.png
  jaccard_macro_pi_HARD_k1.png
  jaccard_macro_pi_HARD_k2.png
  jaccard_macro_logits_EASY_k1.png
  jaccard_macro_logits_EASY_k2.png
  jaccard_macro_logits_MEDIUM_k1.png
  jaccard_macro_logits_MEDIUM_k2.png
  jaccard_macro_logits_HARD_k1.png
  jaccard_macro_logits_HARD_k2.png
  jaccard_micro_pi_EASY_k10.png
  jaccard_micro_pi_EASY_k20.png
  jaccard_micro_pi_EASY_k50.png
  jaccard_micro_pi_MEDIUM_k10.png
  jaccard_micro_pi_MEDIUM_k20.png
  jaccard_micro_pi_MEDIUM_k50.png
  jaccard_micro_pi_HARD_k10.png
  jaccard_micro_pi_HARD_k20.png
  jaccard_micro_pi_HARD_k50.png
  jaccard_micro_logits_EASY_k10.png
  jaccard_micro_logits_EASY_k20.png
  jaccard_micro_logi

In [14]:
def md_table(df, cols=None, floatfmt=".4f"):
    if df is None or len(df) == 0:
        return "_No data_"
    d = df if cols is None else df[cols]
    try:
        return d.to_markdown(index=False, floatfmt=floatfmt)
    except Exception:
        return "```\n" + d.to_string(index=False) + "\n```"


extra_rows = []
for (seed, ckpt_id), (ckpt_dir, ckpt_name) in EXTRA_CHECKPOINTS.items():
    actor, prefix = load_actor(seed, ckpt_name)
    for sc in SCENARIOS:
        st = action_stats(actor, STATES[sc])
        extra_rows.append({
            "seed": seed, "checkpoint": ckpt_name, "scenario": sc,
            "mean_action": st["mean_action"], "frac_action_13": st["frac_action_13"],
            "mean_entropy": st["mean_entropy"], "mean_max_prob": st["mean_max_prob"],
        })
df_extra = pd.DataFrame(extra_rows)
df_extra.to_csv(OUT_DIR / "extra_checkpoint_diagnostics.csv", index=False)

report = []
report.append("# Task 1 - SHAP Feature Importance Stability Across 5 A2C_mod Seeds\n")
report.append(f"Generated: {pd.Timestamp.now():%Y-%m-%d %H:%M:%S}\n")
report.append("## 1. Goal\n")
report.append("Kiem chung SHAP feature importance co on dinh qua 5 lan huan luyen doc lap (seed 42, 123, 256, 512, 1024) hay khong, cho macro (3 nhom) va micro (660 dac trung), tren 3 scenario va 2 target (pi, logits).\n")

cfg_rows = []
for sc, cfg in SCENARIOS.items():
    cfg_rows.append({"scenario": sc, "sales_scale": cfg["sales_scale"], "waste_rate": cfg["waste_rate"]})
report.append("## 2. Configuration\n")
report.append(md_table(pd.DataFrame(cfg_rows)))
report.append(f"\n- N_STATES = {N_STATES}; BG_SIZE = {BG_SIZE}; BG_SEED = {BG_SEED}")
report.append(f"- TARGETS = {TARGETS}; MICRO_TOP_K = {TOP_K}; MACRO_TOP_K = {MACRO_TOP_K}")
report.append("- Micro state: 660-dim [x_init | sales_t*scale | x_init*waste_rate]; macro state: mean of each block.")
report.append("- Actor: 3 -> 32 -> 32 -> 32 -> 14, dropout=0.0; restore only actor with expect_partial().\n")

report.append("## 3. Checkpoints Used\n")
report.append(md_table(df_ckpt[["seed", "checkpoint", "step", "episode", "checkpoint_dir"]]))
report.append("\n## 4. Collapse Diagnostics\n")
report.append(md_table(df_diag[["seed", "scenario", "mean_action", "frac_action_13", "mean_entropy", "mean_max_prob"]]))
report.append("\n### Extra checkpoints\n")
report.append(md_table(df_extra[["seed", "checkpoint", "scenario", "mean_action", "frac_action_13", "mean_entropy", "mean_max_prob"]]))
report.append("\n## 5. Rollout Metrics (100 steps)\n")
report.append(md_table(df_rollout[["seed", "scenario", "reward", "stockout", "waste", "overstock", "quan", "mean_action"]]))

report.append("\n## 6. Macro SHAP (mean +/- SD across seeds)\n")
if len(df_macro_stats):
    for target in TARGETS:
        for scenario in SCENARIOS:
            sub = df_macro_stats[(df_macro_stats["target"] == target) & (df_macro_stats["scenario"] == scenario)]
            report.append(f"### Macro - {target} / {scenario}\n")
            report.append(md_table(sub[["feature", "mean", "sd", "cv"]]))
else:
    report.append("_No macro data_")

report.append("\n## 7. Micro SHAP - Top 20 Features (mean +/- SD across seeds)\n")
if len(df_micro_stats):
    for target in TARGETS:
        for scenario in SCENARIOS:
            sub = df_micro_stats[(df_micro_stats["target"] == target) & (df_micro_stats["scenario"] == scenario)].sort_values("mean", ascending=False).head(20)
            report.append(f"### Micro top-20 - {target} / {scenario}\n")
            report.append(md_table(sub[["feature", "group", "mean", "sd", "cv"]]))
else:
    report.append("_No micro data_")

report.append("\n## 8. Stability Summary\n")
if len(df_summary):
    for level in ["macro", "micro"]:
        for target in TARGETS:
            for scenario in SCENARIOS:
                sub = df_summary[(df_summary["level"] == level) & (df_summary["target"] == target) & (df_summary["scenario"] == scenario)]
                if len(sub) == 0:
                    continue
                report.append(f"### {level} - {target} / {scenario}\n")
                report.append(md_table(sub[["subset", "k", "jaccard_mean", "jaccard_sd", "spearman_all_mean", "spearman_topk_mean", "n_pairs"]]))
else:
    report.append("_No stability data_")

report.append("\n## 9. Cross-target Stability (pi vs logits, same seed)\n")
if len(df_cross):
    report.append(md_table(df_cross[["level", "scenario", "seed", "k", "jaccard", "spearman_all"]]))
else:
    report.append("_No cross-target data_")

report.append("\n## 10. Seed 256 Collapse\n")
s256 = df_diag[df_diag["seed"] == 256]
report.append(md_table(s256[["scenario", "mean_action", "frac_action_13", "mean_entropy", "mean_max_prob"]]))
report.append("\nSeed 256 chon action 13 cho phan lon SKU, entropy gan 0. Ket qua SHAP cua seed 256 can duoc bao cao rieng va khong nen dung mot minh de ket luan.\n")

report.append("\n## 11. Figures\n")
for p in sorted(FIG_DIR.glob("*.png")):
    rel = p.relative_to(OUT_DIR).as_posix()
    report.append(f"![{p.stem}]({rel})\n")

report.append("\n## 12. Key Findings\n")

if len(df_diag):
    report.append("### 12.1. Mức độ bão hòa của policy\n")
    ent_lines = []
    for seed in SEEDS:
        sub = df_diag[df_diag["seed"] == seed]
        if len(sub):
            ent_lines.append(f"seed {seed}: entropy {sub['mean_entropy'].mean():.4f}, action-13 {sub['frac_action_13'].mean():.3f}")
    report.append("- " + "; ".join(ent_lines) + ".")
    report.append("- Cả 5 seed đều gần như deterministic; action mode khác nhau. Seed 256 có tỷ lệ action 13 cao nhất và được báo cáo riêng, nhưng seed 42/512 cũng rất bão hòa.\n")

if len(df_summary):
    report.append("### 12.2. Micro (660 đặc trưng)\n")
    for target in TARGETS:
        sub = df_summary[(df_summary["level"] == "micro") & (df_summary["target"] == target) & (df_summary["k"] == 20)]
        if len(sub) == 0:
            continue
        a = sub[sub["subset"] == "all"]
        e = sub[sub["subset"] == "excl_256"]
        s = sub[sub["subset"] == "stable_pair"]
        j_all = a["jaccard_mean"].mean() if len(a) else float("nan")
        j_ex = e["jaccard_mean"].mean() if len(e) else float("nan")
        j_sp = s["jaccard_mean"].iloc[0] if len(s) else float("nan")
        r_all = a["spearman_all_mean"].mean() if len(a) else float("nan")
        r_ex = e["spearman_all_mean"].mean() if len(e) else float("nan")
        report.append(f"- `{target}`, k=20: Jaccard all={j_all:.3f}, excl_256={j_ex:.3f}, stable_pair={j_sp:.3f}; Spearman all={r_all:.3f}, excl_256={r_ex:.3f}.")
    if len(df_micro_stats):
        top = df_micro_stats[(df_micro_stats["target"] == "logits") & (df_micro_stats["scenario"] == "MEDIUM")].sort_values("mean", ascending=False).head(8)
        if len(top):
            report.append("- Top-8 micro features (logits, MEDIUM): " + ", ".join(top["feature"].tolist()) + ".")
    report.append("- Kết luận micro: **ổn định cao**, đặc biệt với logits (Jaccard = 1.000); π vẫn ổn định nhưng nhạy hơn với seed bão hòa.\n")

    report.append("### 12.3. Macro (3 nhóm)\n")
    for target in TARGETS:
        sub = df_summary[(df_summary["level"] == "macro") & (df_summary["target"] == target) & (df_summary["k"] == 1)]
        if len(sub) == 0:
            continue
        a = sub[sub["subset"] == "all"]
        e = sub[sub["subset"] == "excl_256"]
        s = sub[sub["subset"] == "stable_pair"]
        j_all = a["jaccard_mean"].mean() if len(a) else float("nan")
        j_ex = e["jaccard_mean"].mean() if len(e) else float("nan")
        j_sp = s["jaccard_mean"].iloc[0] if len(s) else float("nan")
        r_all = a["spearman_all_mean"].mean() if len(a) else float("nan")
        r_sp = s["spearman_all_mean"].iloc[0] if len(s) else float("nan")
        report.append(f"- `{target}`, k=1: Jaccard all={j_all:.3f}, excl_256={j_ex:.3f}, stable_pair={j_sp:.3f}; Spearman all={r_all:.3f}, stable_pair={r_sp:.3f}.")
    if len(df_macro_imp):
        for target in TARGETS:
            sub = df_macro_imp[(df_macro_imp["target"] == target) & (df_macro_imp["scenario"] == "MEDIUM")]
            if len(sub) == 0:
                continue
            top1 = sub.loc[sub.groupby("seed")["mean_abs_shap"].idxmax(), ["seed", "feature", "mean_abs_shap"]]
            parts = [f"seed {int(r['seed'])}={r['feature']} ({r['mean_abs_shap']:.4g})" for _, r in top1.iterrows()]
            report.append(f"- Top-1 macro `{target}` (MEDIUM) theo seed: " + "; ".join(parts) + ".")
    report.append("- Kết luận macro: **ổn định có điều kiện**. Logits luôn chọn Demand; π có top-1 dao động giữa Inventory và Demand, đặc biệt seed 512 nghiêng mạnh về Demand.\n")

    report.append("### 12.4. So sánh π vs logits\n")
    if len(df_cross):
        for level in ["macro", "micro"]:
            kk = 1 if level == "macro" else 20
            sub = df_cross[(df_cross["level"] == level) & (df_cross["k"] == kk)]
            if len(sub):
                report.append(f"- `{level}`, k={kk}: Jaccard trung bình={sub['jaccard'].mean():.3f}, Spearman={sub['spearman_all'].mean():.3f}.")
    report.append("- Micro: hai target đồng thuận cao ở mức SKU. Macro: top-1 khác nhau, nhưng top-2 đồng thuận (Jaccard = 1.0).\n")

    report.append("### 12.5. Kết luận tổng\n")
    report.append("- **Micro SHAP ổn định qua 5 seed**, đặc biệt với logits; đây là bằng chứng mạnh cho tính tái lập ở mức SKU.")
    report.append("- **Macro SHAP chỉ ổn định có điều kiện**: logits nhất quán (Demand), π không ổn định ở top-1 do policy saturation và aggregation.")
    report.append("- **Seed 256** nên báo cáo riêng; không nên dùng một mình để kết luận.")
    report.append("- **Khuyến nghị:** dùng logits làm target chính cho stability, π làm target quyết định; báo cáo top-k (k>=10 cho micro, k=2 cho macro) thay vì top-1.\n")
else:
    report.append("_No stability data_")

report.append("\n## 13. Notes\n")
report.append("- Seed 42 chi 530 episode (ckpt-53) vs 600 cua cac seed khac; co the la mot phan nguyen nhan khac biet.")
report.append("- Macro state tong hop lam inventory/waste gan nhu co dinh trong mot scenario; macro SHAP chu yeu phan anh demand.")
report.append("- PartitionExplainer co yeu to ngau nhien; da co dinh seed va luu raw NPZ. Co the bat noise-floor de do variance.")
report.append("- Khong dung lai output SHAP cu vi checkpoint 'output Training' khong trung 5 seed nay.\n")

report_path = OUT_DIR / "Task1_Report.md"
report_path.write_text("\n".join(report), encoding="utf-8")
print("Wrote", report_path)
print("Rows: macro_imp", len(df_macro_imp), "micro_imp", len(df_micro_imp), "pairs", len(df_pairs))


Wrote C:\GitHub\Q-learning-for-Inventory-Management\Feedback 23-11\task1\output\Task1_Report.md
Rows: macro_imp 90 micro_imp 19800 pairs 510


In [15]:
if RUN_MATCHED_500:
    matched = {}
    for seed in SEEDS:
        ckpt_dir, _ = CHECKPOINTS[seed]
        if not (Path(ckpt_dir) / "ckpt-50.index").exists():
            print(f"skip seed {seed}: no ckpt-50")
            continue
        actor, prefix = load_actor(seed, "ckpt-50")
        X = STATES["MEDIUM"]
        masker = shap.maskers.Partition(BG_MICRO, max_samples=BG_SIZE)
        explainer = shap.PartitionExplainer(_micro_probs, masker)
        np.random.seed(BG_SEED)
        sv = explainer(X)
        arr = _normalize_micro(sv.values if hasattr(sv, "values") else sv)
        imp = mean_abs_micro(arr)
        matched[seed] = imp
        np.savez_compressed(RAW_DIR / f"micro_pi_MEDIUM_matched500_seed{seed}.npz", values=arr, X=X, background=BG_MICRO)
        print("matched500", seed, "done")
    rows = []
    for i, sa in enumerate([s for s in SEEDS if s in matched]):
        for sb in [s for s in SEEDS if s in matched][i + 1:]:
            for k in TOP_K:
                rows.append({
                    "seed_a": sa, "seed_b": sb, "k": k,
                    "jaccard": jaccard(topk_idx(matched[sa], k), topk_idx(matched[sb], k)),
                    "spearman_all": _safe_spearman(matched[sa], matched[sb]),
                })
    df_matched = pd.DataFrame(rows)
    df_matched.to_csv(OUT_DIR / "matched500_stability.csv", index=False)
    display(df_matched)

if RUN_NOISE_FLOOR:
    load_actor(42)
    X = STATES_MACRO["MEDIUM"]
    explainer = shap.KernelExplainer(_macro_pi, BG_MACRO)
    imps = []
    for rep in range(5):
        np.random.seed(1000 + rep)
        sv = explainer.shap_values(X)
        imps.append(mean_abs_macro(_normalize_macro(sv, len(X))))
    rows = []
    for i in range(len(imps)):
        for j in range(i + 1, len(imps)):
            rows.append({
                "rep_a": i, "rep_b": j,
                "spearman_all": _safe_spearman(imps[i], imps[j]),
                "jaccard_k1": jaccard(topk_idx(imps[i], 1), topk_idx(imps[j], 1)),
            })
    df_noise = pd.DataFrame(rows)
    df_noise.to_csv(OUT_DIR / "noise_floor_macro.csv", index=False)
    display(df_noise)
